# EDA — Classificador de Categorias de Notícias

Análise exploratória do acervo de notícias da Folha de S.Paulo (2015–2017).

Objetivo (RF1 do PRD): entender a **distribuição das categorias**, as **características dos títulos** e o **período** do dataset, para embasar as decisões de modelagem.

> Os dados (`data/articles.csv`, ~480 MB) não estão no repositório. Veja `data/README.md`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None) #pra não truncar as colunas 


SEED = 42
np.random.seed(SEED)

In [2]:
#lendo os dados
CAMINHO_DADOS = "data/articles.csv"

df = pd.read_csv(CAMINHO_DADOS)
print(f"Registros: {len(df):,}")
print(f"Colunas:   {list(df.columns)}")
df.head()

Registros: 167,053
Colunas:   ['title', 'text', 'date', 'category', 'subcategory', 'link']


,title,text,date,category,subcategory,link
0,"Lula diz que está 'lascado', mas que ainda tem...",Com a possibilidade de uma condenação impedir ...,2017-09-10,poder,NaN,http://www1.folha.uol.com.br/poder/2017/10/192...
1,"'Decidi ser escrava das mulheres que sofrem', ...","Para Oumou Sangaré, cantora e ativista malines...",2017-09-10,ilustrada,NaN,http://www1.folha.uol.com.br/ilustrada/2017/10...
2,Três reportagens da Folha ganham Prêmio Petrob...,Três reportagens da Folha foram vencedoras do ...,2017-09-10,poder,NaN,http://www1.folha.uol.com.br/poder/2017/10/192...
3,Filme 'Star Wars: Os Últimos Jedi' ganha trail...,A Disney divulgou na noite desta segunda-feira...,2017-09-10,ilustrada,NaN,http://www1.folha.uol.com.br/ilustrada/2017/10...
4,CBSS inicia acordos com fintechs e quer 30% do...,"O CBSS, banco da holding Elopar dos sócios Bra...",2017-09-10,mercado,NaN,http://www1.folha.uol.com.br/mercado/2017/10/1...


In [3]:
# Estrutura, tipos e nulos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167053 entries, 0 to 167052
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   title        167053 non-null  object
 1   text         166288 non-null  object
 2   date         167053 non-null  object
 3   category     167053 non-null  object
 4   subcategory  29635 non-null   object
 5   link         167053 non-null  object
dtypes: object(6)
memory usage: 7.6+ MB


In [4]:
# Nulos por coluna
nulos = df.isna().sum()
pct = (nulos / len(df) * 100).round(2)
pd.DataFrame({"nulos": nulos, "%": pct})


,nulos,%
title,0,0.00
text,765,0.46
date,0,0.00
category,0,0.00
subcategory,137418,82.26
link,0,0.00


## Distribuição das categorias



In [5]:
contagem = df["category"].value_counts()
print(f"Total de categorias: {contagem.shape[0]}")
contagem

# Aqui observamos categorias como "musica; 2016; bichos; 2015". Todas com 1 registro. 
# Suponho que os registros de categoria foram feitos a partir da URL, e para validar faço a verificação.

Total de categorias: 48


category
poder                           22022
colunas                         21622
mercado                         20970
esporte                         19730
mundo                           17130
cotidiano                       16967
ilustrada                       16345
opiniao                          4525
paineldoleitor                   4011
saopaulo                         3955
tec                              2260
tv                               2142
educacao                         2118
turismo                          1903
ilustrissima                     1411
ciencia                          1335
equilibrioesaude                 1312
sobretudo                        1057
bbc                               980
folhinha                          876
empreendedorsocial                841
comida                            828
asmais                            548
ambiente                          491
seminariosfolha                   379
serafina                          334
o-m

In [10]:
print(f"Total das últimas categorias: ")
contagem.tail(10)

Total das últimas categorias: 


category
treinamento                 21
treinamentocienciaesaude    18
mulher                      16
euronews                     8
ombudsman                    3
contas-de-casa               2
musica                       1
2016                         1
bichos                       1
2015                         1
Name: count, dtype: int64

In [ ]:
# Títulos das 10 categorias com menos registros (para inspeção manual).
# Buscamos no df ORIGINAL, pois essas categorias raras serão removidas do df_modelo mais adiante.
menores_categorias = contagem.tail(10).index

titulos_raros = (
    df.loc[df["category"].isin(menores_categorias), ["category", "title"]]
    .sort_values("category")
    .reset_index(drop=True)
)
print(f"Total de títulos nas 10 menores categorias: {len(titulos_raros)}")

# Exporta para Excel para leitura dos títulos
ARQUIVO_XLSX = "categorias_raras_titulos.xlsx"
titulos_raros.to_excel(ARQUIVO_XLSX, index=False)
print(f"Exportado para: {ARQUIVO_XLSX}")

titulos_raros

In [ ]:
from urllib.parse import urlparse


def secao_da_url(link: str) -> str | None:
    #Extrai o primeiro trecho do caminho da URL (a 'seção' da Folha).
    #Ex.: 'http://www1.folha.uol.com.br/poder/2017/10/123-...' -> 'poder'.
    #Retorna None se não houver caminho utilizável.
    
    if not isinstance(link, str):
        return None
    caminho = urlparse(link).path.strip("/")
    if not caminho:
        return None
    return caminho.split("/")[0].lower()


df["secao_url"] = df["link"].map(secao_da_url)

# A seção da URL bate com a categoria?
df["bate"] = df["secao_url"] == df["category"].str.lower()

total = len(df)
iguais = int(df["bate"].sum())
print(f"Seção da URL == category : {iguais:,} de {total:,} ({iguais / total:.2%})")
print(f"Divergências             : {total - iguais:,} ({1 - iguais / total:.2%})")
# E aqui confirmamos que a seção da URL bate com a categoria por completo. Ou seja, a categoria foi extraída da URL.

Seção da URL == category : 167,053 de 167,053 (100.00%)
Divergências             : 0 (0.00%)


In [ ]:
# Caso divergem >> Onde diverge: quais pares (category -> secao_url) aparecem?
divergentes = df.loc[~df["bate"], ["category", "secao_url"]]
print(f"Registros divergentes: {len(divergentes):,}\n")

# Pares mais comuns de divergência
pares = (
    divergentes.value_counts(["category", "secao_url"])
    .rename("qtd")
    .reset_index()
)
pares.head(30)

Registros divergentes: 0



,category,secao_url,qtd


## Corte de categorias raras (suporte mínimo)

A distribuição é long-tail: as 7 maiores categorias cobrem ~81% dos dados, enquanto
dezenas de categorias no rabo têm pouquíssimos exemplos (algumas com 1 registro).

Duas razões para cortar as raras antes de treinar:
1. **Restrição do split estratificado** : classes com < 2 exemplos quebram o `train_test_split`
   estratificado (não há como colocar 1 exemplo em treino *e* teste).
2. **Qualidade e avaliação** : com pouquíssimos exemplos o modelo não aprende o padrão da classe,
   e a métrica por classe fica sem significado estatístico.

O limiar abaixo (`MIN_EXEMPLOS`) foi parametrizável para ver a densidade de dados mantidos

In [14]:
# --- Parâmetro do corte: mude e re-execute para testar cenários ---
MIN_EXEMPLOS = 50

contagem = df["category"].value_counts()

mantidas = contagem[contagem >= MIN_EXEMPLOS]
removidas = contagem[contagem < MIN_EXEMPLOS]

regs_mantidos = int(mantidas.sum())
regs_removidos = int(removidas.sum())
total = len(df)

print(f"Limiar (MIN_EXEMPLOS)        : {MIN_EXEMPLOS}")
print(f"Categorias mantidas          : {mantidas.shape[0]} de {contagem.shape[0]}")
print(f"Categorias removidas         : {removidas.shape[0]}")
print(f"Registros mantidos           : {regs_mantidos:,} ({regs_mantidos / total:.2%})")
print(f"Registros removidos          : {regs_removidos:,} ({regs_removidos / total:.3%})")
print(f"\nCategorias removidas com este limiar:")
print(removidas.to_string())

Limiar (MIN_EXEMPLOS)        : 50
Categorias mantidas          : 31 de 48
Categorias removidas         : 17
Registros mantidos           : 166,720 (99.80%)
Registros removidos          : 333 (0.199%)

Categorias removidas com este limiar:
category
dw                              48
infograficos                    43
cenarios-2017                   43
especial                        43
rfi                             29
guia-de-livros-filmes-discos    28
multimidia                      27
treinamento                     21
treinamentocienciaesaude        18
mulher                          16
euronews                         8
ombudsman                        3
contas-de-casa                   2
musica                           1
2016                             1
bichos                           1
2015                             1


In [17]:
# Comparação de cenários: efeito de vários limiares lado a lado
linhas = []
for limiar in [50, 100, 250, 500, 1000]:
    mant = contagem[contagem >= limiar]
    linhas.append({
        "limiar": limiar,
        "categorias_mantidas": mant.shape[0],
        "categorias_removidas": contagem.shape[0] - mant.shape[0],
        "registros_mantidos": int(mant.sum()),
        "%_dados_mantidos": round(mant.sum() / total * 100, 3),
    })

pd.DataFrame(linhas)

,limiar,categorias_mantidas,categorias_removidas,registros_mantidos,%_dados_mantidos
0,50,31,17,166720,99.801
1,100,29,19,166570,99.711
2,250,26,22,166092,99.425
3,500,23,25,164888,98.704
4,1000,18,30,160815,96.266


In [18]:
# Aplica o corte -> dataframe de trabalho para a modelagem
categorias_validas = contagem[contagem >= MIN_EXEMPLOS].index

df_modelo = df[df["category"].isin(categorias_validas)].copy()

print(f"df original : {len(df):,} registros, {df['category'].nunique()} categorias")
print(f"df_modelo   : {len(df_modelo):,} registros, {df_modelo['category'].nunique()} categorias")
print(f"\nMenor categoria retida: '{df_modelo['category'].value_counts().index[-1]}' "
      f"({df_modelo['category'].value_counts().iloc[-1]} exemplos)")

df original : 167,053 registros, 48 categorias
df_modelo   : 166,720 registros, 31 categorias

Menor categoria retida: 'banco-de-dados' (64 exemplos)


## Seleção de colunas e pré-processamento dos títulos

A modelagem usa apenas **`title`** (feature) e **`category`** (alvo). As demais colunas são
descartadas: `text` e `date` estão fora do escopo, `subcategory` tem baixa cobertura, e
`link`/`secao_url`/`bate` foram apenas instrumentos de auditoria (o `link` vaza o rótulo).

In [ ]:
# Mantém apenas o essencial para a modelagem
df_modelo = df_modelo[["title", "category"]].reset_index(drop=True)
print(f"df_modelo: {len(df_modelo):,} linhas, colunas = {list(df_modelo.columns)}")
df_modelo.head()

In [ ]:
# Carrega o modelo de português do spaCy (PRD, seção 7).
# Desabilitamos parser e NER: para lematizar títulos só precisamos de tokenizer + tagger + lemmatizer.
import spacy

MODELO_SPACY = "pt_core_news_sm"

try:
    nlp = spacy.load(MODELO_SPACY, disable=["parser", "ner"])
except OSError:
    # Fallback: baixa o modelo se ainda não estiver instalado (ex.: primeira execução no Colab)
    from spacy.cli import download
    download(MODELO_SPACY)
    nlp = spacy.load(MODELO_SPACY, disable=["parser", "ner"])

print(f"Modelo carregado: {MODELO_SPACY}")
print(f"Pipeline ativo  : {nlp.pipe_names}")

In [ ]:
def preprocessar(doc) -> str:
    """Converte um Doc do spaCy em uma string limpa para o TF-IDF.

    Regras (PRD, seção 7 — lematização sem stemming agressivo):
      - mantém apenas tokens alfabéticos (descarta números, pontuação, símbolos);
      - remove stopwords em português;
      - usa o lema em minúsculas (nomes próprios são preservados como lema, não sofrem stemming).
    """
    return " ".join(
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and not token.is_stop
    )


# nlp.pipe processa em lote (muito mais rápido que aplicar por linha).
# Em ~165 mil títulos, leva alguns minutos.
titulos = df_modelo["title"].astype(str).tolist()
df_modelo["title_proc"] = [preprocessar(doc) for doc in nlp.pipe(titulos, batch_size=1000)]

print("Pré-processamento concluído.")
df_modelo[["title", "title_proc"]].head(10)

In [ ]:
# Títulos que ficaram vazios após o pré-processamento (só tinham números/stopwords/pontuação)
vazios = df_modelo["title_proc"].str.strip() == ""
print(f"Títulos vazios após pré-processamento: {int(vazios.sum()):,} ({vazios.mean():.3%})")

if vazios.any():
    display(df_modelo.loc[vazios, ["title", "category"]].head(10))

# Remove os vazios — não têm sinal para o modelo aprender
df_modelo = df_modelo[~vazios].reset_index(drop=True)
print(f"\ndf_modelo final: {len(df_modelo):,} linhas")

## Split estratificado 80/20

Disciplina anti-vazamento (o recado central do vídeo do Let's Data): **o split vem antes de
qualquer transformação aprendida dos dados**. O TF-IDF aprende o vocabulário e os pesos IDF —
isso é "conhecimento" que só pode vir do **treino**. Se ajustássemos o TF-IDF na base inteira,
o vocabulário do teste vazaria para dentro do modelo e a avaliação ficaria otimista demais.

Fazemos o `train_test_split` com `stratify=y` aqui, e a vetorização passa a viver **dentro do
Pipeline** (seção seguinte) — assim o `fit` do TF-IDF acontece só sobre o treino, de forma
automática e à prova de vazamento.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_modelo["title_proc"]   # feature: título já pré-processado
y = df_modelo["category"]     # alvo: categoria

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,          # mantém a proporção das categorias nos dois conjuntos
    random_state=SEED,   # reprodutibilidade (PRD, seção 6)
)

print(f"Treino: {X_train.shape[0]:,} títulos")
print(f"Teste : {X_test.shape[0]:,} títulos")

# Confere que a estratificação preservou as proporções (treino vs teste)
proporcoes = pd.DataFrame({
    "treino_%": y_train.value_counts(normalize=True).mul(100).round(2),
    "teste_%": y_test.value_counts(normalize=True).mul(100).round(2),
})
proporcoes.head(10)

## Modelo baseline — Pipeline TF-IDF + Regressão Logística

Unimos vetorização e classificador num único `Pipeline` do sklearn. Vantagens:

- **Anti-vazamento automático**: `pipeline.fit(X_train)` ajusta o TF-IDF só no treino.
- **`class_weight="balanced"`**: reponderar as classes pelo inverso da frequência neutraliza o
  viés de densidade (as 7 categorias gigantes deixam de engolir as médias) — sem inventar dados,
  ao contrário do SMOTE (que descartamos por não caber em texto esparso/minoria-artefato).
- **Serialização limpa**: o Pipeline inteiro vira um `.joblib` único que a API carrega e aplica
  direto no título cru.

Parâmetros do TF-IDF (baseline): `ngram_range=(1,2)`, `min_df=5`, `max_df=0.9`, `sublinear_tf=True`.
As stopwords já foram removidas no pré-processamento com spaCy.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.9,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",   # neutraliza o desbalanceamento
        n_jobs=-1,
        random_state=SEED,
    )),
])

pipeline

In [ ]:
import time

# Treina o pipeline inteiro. O TF-IDF é ajustado SÓ com X_train (anti-vazamento).
t0 = time.time()
pipeline.fit(X_train, y_train)
print(f"Treino concluído em {time.time() - t0:.0f}s")

# Inspeção do vocabulário aprendido (acessível via named_steps)
vocab = pipeline.named_steps["tfidf"].vocabulary_
print(f"Tamanho do vocabulário: {len(vocab):,} termos")

## Avaliação do modelo

Avaliamos sobre o teste **real** (não rebalanceado):

- **Acurácia**: acerto global (dominada pelas classes grandes).
- **F1 macro**: média simples do F1 por classe — trata toda categoria igual, então **penaliza**
  se o modelo abandonar as categorias médias/pequenas. É a métrica-chave dado o desbalanceamento.
- **F1 weighted**: F1 ponderado pelo tamanho de cada classe — reflete o desempenho no volume real.
- **`classification_report`**: precisão/recall/F1 por categoria, para ver *onde* o modelo erra.

A distância entre **macro** e **weighted** é, por si só, a medida do impacto do desbalanceamento.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_pred = pipeline.predict(X_test)

print(f"Acurácia    : {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 macro    : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1 weighted : {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("\n=== Relatório por categoria ===")
print(classification_report(y_test, y_pred, zero_division=0))

## Serialização do pipeline

Salvamos o Pipeline treinado (TF-IDF + classificador) num único arquivo `.joblib`. Este é o
**elo entre a Fase 1 (treino) e a Fase 2 (API)** do PRD: a API carrega este artefato uma vez na
inicialização e, a cada requisição, aplica `predict` direto no título cru — sem re-treinar.

> **Importante:** o título recebido pela API precisa passar pelo **mesmo pré-processamento**
> (spaCy) antes do `predict`, já que o modelo foi treinado sobre `title_proc`. Isso será tratado
> na implementação da API.

In [ ]:
import joblib
from pathlib import Path

Path("modelo").mkdir(exist_ok=True)
CAMINHO_MODELO = "modelo/pipeline.joblib"
joblib.dump(pipeline, CAMINHO_MODELO)

print(f"Pipeline salvo em: {CAMINHO_MODELO}")

# Teste rápido de sanidade: recarrega e classifica um título novo
modelo_carregado = joblib.load(CAMINHO_MODELO)
exemplo = "Seleção brasileira vence e se classifica para a final da Copa"
# Obs.: em produção o título passa pelo preprocessar(); aqui é só um smoke test
print(f"\nTítulo de teste : {exemplo!r}")
print(f"Categoria prevista: {modelo_carregado.predict([exemplo])[0]!r}")